# Random Walk Metropolis-Hastings — Research Demo

This notebook demonstrates the full Bayesian inference workflow using the RWMH sampler:
1. Define a model (likelihood + prior → unnormalized posterior)
2. Run the sampler with multiple chains, burn-in, and thinning
3. Diagnose convergence (trace, ACF, ESS, Gelman-Rubin R-hat)
4. Interpret the posterior

Two models are covered:
- **Normal Mean** — conjugate-ish, easy target (baseline)
- **Bimodal Mixture** — hard target, shows how proposal width matters

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.sampler      import metropolis_hastings
from src.targets      import make_normal_mean_target, cauchy_prior_pdf, bimodal_log_target, bimodal_true_pdf
from src.diagnostics  import effective_sample_size, gelman_rubin, chain_summary
from src.plots        import (trace_plot, acf_plot, posterior_density_plot,
                               convergence_plot, multi_chain_trace)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

---
## Part 1 — Normal Mean Model

$$
y_i \mid \mu \sim \mathcal{N}(\mu, 1), \quad i = 1,\ldots,n
\qquad
\mu \sim \text{Cauchy}(0, 1)
$$

Log-posterior (up to constant):
$$
\log \pi(\mu \mid y) \propto n\left(\bar{y}\,\mu - \tfrac{1}{2}\mu^2\right) - \log(1 + \mu^2)
$$

In [ ]:
y = np.array([1.2, 1.4, -0.5, 0.3, 0.9, 2.3, 1.0, 0.1, 1.3, 1.9])
y_bar, n = y.mean(), len(y)
print(f'n={n},  ȳ={y_bar:.4f},  s={y.std():.4f}')

In [ ]:
log_target = make_normal_mean_target(n, y_bar)

result = metropolis_hastings(
    log_target = log_target,
    init       = 30.0,       # far initial value — tests burn-in
    n_iter     = 5_000,
    cand_std   = y.std(),
    burn_in    = 500,
    thin       = 1,
    n_chains   = 4,
    seed       = 42,
)

chains      = result['samples']    # shape (4, 4500)
all_samples = chains.flatten()
print(f'Shape: {chains.shape}')
print(f'Acceptance rates: {result["acceptance_rates"].round(3)}')

### Convergence diagnostics

In [ ]:
stats = chain_summary(chains)

In [ ]:
# Trace plot — chain 1
samples = np.asarray(chains[0])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(samples, lw=0.7, color='steelblue', alpha=0.9)
ax.axhline(samples.mean(), color='crimson', lw=1.2, linestyle='--',
           label=f'mean={samples.mean():.3f}')
ax.set(xlabel='Iteration', ylabel='μ', title='Trace — Normal Mean Model (chain 1)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Running mean — all 4 chains
iters = np.arange(1, chains.shape[1] + 1)
fig, ax = plt.subplots(figsize=(10, 3))
for i, c in enumerate(chains):
    ax.plot(iters, np.cumsum(c) / iters, lw=1, alpha=0.8, label=f'Chain {i+1}')
ax.axhline(y_bar, color='crimson', lw=1.4, linestyle='--', label=f'ȳ={y_bar:.3f}')
ax.set(xlabel='Iteration', ylabel='Running mean', title='Convergence — Running Mean')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ACF plot
from src.diagnostics import autocorrelation
acf  = autocorrelation(chains[0], max_lag=60)
lags = np.arange(len(acf))
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(lags, acf, color='steelblue', alpha=0.75, width=0.8)
bound = 1.96 / np.sqrt(len(chains[0]))
ax.axhline(bound,  color='crimson', lw=1, linestyle='--', label='95% CI')
ax.axhline(-bound, color='crimson', lw=1, linestyle='--')
ax.set(xlabel='Lag', ylabel='ACF', title='Autocorrelation — Normal Mean (chain 1)')
ax.legend(); plt.tight_layout(); plt.show()

### Posterior summary

In [ ]:
import pandas as pd
from scipy.stats import t as t_dist

ci_lo, ci_hi = np.percentile(all_samples, [2.5, 97.5])
print(f'Posterior mean : {all_samples.mean():.4f}')
print(f'Posterior std  : {all_samples.std():.4f}')
print(f'95% credible interval: [{ci_lo:.4f}, {ci_hi:.4f}]')

xs = np.linspace(-1, 3, 400)
fig, ax = plt.subplots(figsize=(10, 4))
pd.Series(all_samples).plot.density(ax=ax, color='seagreen', lw=2, label='Posterior')
ax.plot(xs, t_dist.pdf(xs, df=1), color='royalblue', lw=1.5, linestyle='--', label='Prior (Cauchy)')
ax.axvline(y_bar, color='crimson', lw=1.5, linestyle='-', label=f'ȳ={y_bar:.3f}')
ax.set(xlabel='μ', ylabel='Density', title='Prior vs Posterior — Normal Mean Model', xlim=(-1, 3))
ax.legend(); plt.tight_layout(); plt.show()

---
## Part 2 — Bimodal Mixture (Proposal Width Study)

$$
p(x) = 0.5 \cdot \mathcal{N}(x;\,-2,\,0.49) + 0.5 \cdot \mathcal{N}(x;\,+2,\,0.49)
$$

The two modes are 4 units apart. We compare:
- **Narrow** proposal (σ = 0.5): high acceptance, but chain can get stuck in one mode
- **Wide** proposal (σ = 3.0): crosses the valley between modes, better mixing

In [ ]:
def run_mixture(cand_std, label):
    result = metropolis_hastings(
        log_target = bimodal_log_target,
        init       = 0.0,
        n_iter     = 10_000,
        cand_std   = cand_std,
        burn_in    = 1_000,
        thin       = 2,
        n_chains   = 4,
        seed       = 42,
    )
    chains = result['samples']
    print(f'\n--- {label} (σ={cand_std}) ---')
    print(f'Acceptance rates: {result["acceptance_rates"].round(3)}')
    chain_summary(chains)
    return result

r_narrow = run_mixture(0.5, 'Narrow proposal')
r_wide   = run_mixture(3.0, 'Wide proposal')

In [ ]:
xs = np.linspace(-5, 5, 400)
true_density = bimodal_true_pdf(xs)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, result, label in zip(axes,
                              [r_narrow, r_wide],
                              ['Narrow σ=0.5', 'Wide σ=3.0']):
    samples = result['samples'].flatten()
    pd.Series(samples).plot.density(ax=ax, color='seagreen', lw=2, label='Sampled posterior')
    ax.plot(xs, true_density, color='royalblue', lw=1.5, linestyle='--', label='True density')
    ax.set(xlabel='x', title=f'Bimodal Mixture — {label}', xlim=(-5, 5))
    ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# Trace comparison — chain 1 for each proposal width
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, result, label in zip(axes,
                              [r_narrow, r_wide],
                              ['Narrow σ=0.5', 'Wide σ=3.0']):
    ax.plot(result['samples'][0], lw=0.6, color='steelblue', alpha=0.9)
    ax.set_ylabel(label, fontsize=9)
    ax.axhline(0, color='gray', lw=0.5, linestyle=':')

axes[-1].set_xlabel('Iteration (post burn-in, thinned)')
fig.suptitle('Trace — Bimodal Mixture: Narrow vs Wide Proposal')
plt.tight_layout(); plt.show()

---
## Key takeaways

| | Normal Mean | Bimodal (narrow) | Bimodal (wide) |
|---|---|---|---|
| Acceptance rate | ~0.5–0.7 | ~0.8–0.9 | ~0.3–0.5 |
| Both modes visited | n/a | No | Yes |
| R-hat < 1.1 | Yes | Often not | Yes |
| ESS / n | > 30% | < 10% | > 20% |

**Rule of thumb**: for a unimodal target, aim for an acceptance rate of ~0.44 (optimal for 1-D Normal). For multimodal targets, you may need a wider proposal, parallel tempering, or other advanced techniques.

See `examples/normal_mean.py` and `examples/mixture.py` for standalone scripts that save all plots to `outputs/`.